# Focus Area 4 — Agro-Ecological Zoning with PyAEZ

This notebook bridges the climate monitoring work from Focus Areas 1–3 into agricultural interpretation using **PyAEZ v2.2** (FAO / AIT).

**What it computes:**
- **Thermal Climate Classification** — which climate zones exist in the country
- **Length of Growing Period (LGP)** — how many days per year support crop growth
- **Agro-Ecological Zone (AEZ) map** — spatial distribution of zones
- **Crop suitability summary** — which crops are viable and where

**Inputs drawn from Shared Drive (cached by FA1–FA3):**
- ERA5 monthly temperature (tmin, tavg, tmax) — from FA3
- Best-scoring satellite precipitation (CHIRPS/TAMSAT/ERA5/IMERG) — from FA2
- PET (Hargreaves) — recomputed here from temperature

---

In [ ]:
# @title ### 0) Configuration {"display-mode":"form"}
# @markdown Set the country and which satellite product to use for precipitation.
# @markdown The precipitation product should be the one that scored highest in Focus Area 2.

import os
import numpy as np
from datetime import date

# ── COUNTRY ───────────────────────────────────────────────────────────────────
COUNTRY = 'Ethiopia'    # 'Ethiopia' | 'Tanzania' | 'Eritrea' | 'Djibouti'

COUNTRY_BBOX = {
    'Ethiopia':  [33.0,  3.0, 48.0, 15.0],
    'Tanzania':  [29.0, -12.0, 41.0, -1.0],
    'Eritrea':   [36.5,  12.5, 43.5, 18.0],
    'Djibouti':  [41.5,  10.9, 43.5, 12.7],
}
xmin, ymin, xmax, ymax = COUNTRY_BBOX[COUNTRY]

# ── PRECIPITATION SOURCE ──────────────────────────────────────────────────────
# Set to whichever product scored highest in FA2 for this country.
# Options: 'CHIRPS' | 'TAMSAT' | 'ERA5' | 'IMERG'
PRECIP_SOURCE = 'CHIRPS'

# ── DATE RANGE FOR LONG-TERM ANALYSIS ────────────────────────────────────────
# Uses the long-term ERA5 from FA3 (monthly 2020–2024 by default).
LT_START = 2020
LT_END   = 2024

# ── LOCAL DIRS ────────────────────────────────────────────────────────────────
LOCAL_CACHE_DIR = f'./Datasets/{COUNTRY}'
RESULTS_DIR     = f'./outputs/FA4_AEZ_{COUNTRY}_{date.today().isoformat()}'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,     exist_ok=True)

print(f'✅ CONFIG set.')
print(f'   Country         : {COUNTRY}')
print(f'   Bbox            : [{xmin}, {ymin}, {xmax}, {ymax}]')
print(f'   Precip source   : {PRECIP_SOURCE}')
print(f'   Long-term period: {LT_START}–{LT_END}')
print(f'   Outputs         : {RESULTS_DIR}')

In [ ]:
# @title ### 1) GEE Authentication {"display-mode":"form"}
# @markdown Same service account used across all Focus Area notebooks.

# @title ### 1) GEE Authentication {"display-mode":"form"}

import ee
import json
import gdown

# ── Service account key — downloaded from Shared Drive at runtime ──────────────
# File ID from: https://drive.google.com/file/d/181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT
SA_KEY_FILE_ID = '181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT'
SA_KEY_PATH    = 'service_account_key.json'

print('🔑 Downloading service account key...')
gdown.download(
    f'https://drive.google.com/uc?id={SA_KEY_FILE_ID}',
    SA_KEY_PATH,
    quiet=True
)

if not os.path.exists(SA_KEY_PATH) or os.path.getsize(SA_KEY_PATH) == 0:
    raise RuntimeError(
        '❌ Key file download failed.\n'
        '   → Make sure the file is shared as "Anyone with the link" in Google Drive.'
    )

with open(SA_KEY_PATH) as f:
    _key = json.load(f)

credentials = ee.ServiceAccountCredentials(
    email=_key['client_email'],
    key_file=SA_KEY_PATH
)
ee.Initialize(credentials)
print(f'✅ GEE authenticated as: {_key["client_email"]}')

In [ ]:
# @title ### 2) Install dependencies & mount Shared Drive {"display-mode":"form"}

print('Installing dependencies...')

# GDAL must be installed via apt first, then pip binds to the system library
!apt-get install -y libgdal-dev gdal-bin > /dev/null 2>&1
!pip install gdal[numpy]=="$(gdal-config --version).*" -q
!pip install pyaez==2.2 numba scipy -q
!pip install xarray rioxarray rasterio cartopy -q
print('✅ Dependencies installed.')

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, io, json, shutil
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import date
from google.colab import drive

# PyAEZ modules
try:
    from pyaez import ClimateRegime
    from pyaez import UtilityCalculations
    print('✅ PyAEZ imported successfully.')
    PYAEZ_AVAILABLE = True
except ImportError as e:
    print(f'⚠️  PyAEZ import failed: {e}')
    print('   Analysis will use equivalent manual calculations.')
    PYAEZ_AVAILABLE = False

# ── MOUNT SHARED DRIVE ────────────────────────────────────────────────────────
print('\n📂 Mounting Google Drive...')
drive.mount('/content/drive', force_remount=False)

SHARED_DRIVE_ROOT  = '/content/drive/Shareddrives/NOAA-workshop2'
DATASETS_ROOT      = os.path.join(SHARED_DRIVE_ROOT, 'Datasets')
COUNTRY_DRIVE_DIR  = os.path.join(DATASETS_ROOT, COUNTRY)

if not os.path.exists(SHARED_DRIVE_ROOT):
    print('❌ Shared Drive not found. Check your access to NOAA-workshop2.')
    DRIVE_CACHE_AVAILABLE = False
else:
    os.makedirs(COUNTRY_DRIVE_DIR, exist_ok=True)
    DRIVE_CACHE_AVAILABLE = True
    print(f'✅ Shared Drive ready: {COUNTRY_DRIVE_DIR}')

def try_load_from_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return False
    src = os.path.join(COUNTRY_DRIVE_DIR, filename)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
        shutil.copy(src, local_path)
        print(f'   ✅ Loaded from Shared Drive: {filename}')
        return True
    return False

def cache_to_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return
    dest = os.path.join(COUNTRY_DRIVE_DIR, filename)
    try:
        shutil.copy(local_path, dest)
        print(f'   ☁️  Cached to Shared Drive: {filename}')
    except Exception as e:
        print(f'   ⚠️  Cache failed: {e}')

In [ ]:
# @title ### 3) Load climate inputs from Shared Drive {"display-mode":"form"}
# @markdown Reads the long-term ERA5 temperature (produced by FA3 Step 7)
# @markdown and the best satellite precipitation (produced by FA2) from the Shared Drive.
# @markdown If either is missing, pulls from GEE and caches for future runs.

import ee

# ── HELPER: subset xarray to country bbox ────────────────────────────────────
def subset_to_bbox(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower() or c == 'y'), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower() or c == 'x'), None)
    if lat_dim is None or lon_dim is None:
        return ds
    lv = ds[lat_dim].values
    ls = slice(ymax, ymin) if lv[0] > lv[-1] else slice(ymin, ymax)
    return ds.sel({lat_dim: ls, lon_dim: slice(xmin, xmax)})

# ─────────────────────────────────────────────────────────────────────────────
# A) LONG-TERM ERA5 TEMPERATURE (monthly, from FA3 Step 7)
# Expected Shared Drive filename: era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc
# ─────────────────────────────────────────────────────────────────────────────
_lt_filename  = f'era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc'
_lt_local     = os.path.join(LOCAL_CACHE_DIR, _lt_filename)

era5_lt = None
print(f'\n🔍 Loading long-term ERA5 temperature: {_lt_filename}')

if os.path.exists(_lt_local):
    print('   ✅ Found locally.')
    era5_lt = xr.open_dataset(_lt_local)
elif try_load_from_drive(_lt_local, _lt_filename):
    era5_lt = xr.open_dataset(_lt_local)
else:
    print(f'   ⚠️  Not found — pulling from GEE (this may take 10–15 min)...')
    # Pull monthly ERA5 Land temperature (same approach as FA3 Step 7)
    try:
        _bbox  = [xmin, ymin, xmax, ymax]
        _parts = []
        for yr in range(LT_START, LT_END + 1):
            print(f'   Pulling {yr}...')
            col = (
                ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(f'{yr}-01-01', f'{yr}-12-31')
                .select('temperature_2m')
            )
            region = ee.Geometry.BBox(*_bbox)
            imgs   = col.toList(12)
            yr_arrays = []
            for m in range(min(12, imgs.size().getInfo())):
                img  = ee.Image(imgs.get(m))
                data = img.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=10000,
                    maxPixels=1e9
                ).getInfo()
                t_c  = data.get('temperature_2m', np.nan)
                if t_c is not None and t_c > 150:
                    t_c -= 273.15   # Kelvin → Celsius
                yr_arrays.append(t_c)
            _parts.append(yr_arrays)

        # Build a simple xarray Dataset from the mean values
        # (spatial mean per month — sufficient for LGP / thermal zone)
        months  = pd.date_range(f'{LT_START}-01', periods=len(_parts) * 12, freq='MS')
        vals    = np.array([v for yr in _parts for v in yr], dtype=float)
        era5_lt = xr.Dataset(
            {'temperature_2m': ('time', vals)},
            coords={'time': months}
        )
        era5_lt.to_netcdf(_lt_local)
        cache_to_drive(_lt_local, _lt_filename)
        print('   ✅ Pulled and cached.')
    except Exception as e:
        print(f'   ❌ GEE pull failed: {e}')
        era5_lt = None

if era5_lt is not None:
    print(f'   Loaded: {era5_lt}')

# ─────────────────────────────────────────────────────────────────────────────
# B) SATELLITE PRECIPITATION — best product from FA2
# Expected Shared Drive filename: CHIRPS_{start_date}_{end_date}_{COUNTRY}.nc
# (or TAMSAT / ERA5 / IMERG — same naming convention used by FA2)
# ─────────────────────────────────────────────────────────────────────────────
_precip_filename = None
precip_ds        = None

print(f'\n🔍 Searching for {PRECIP_SOURCE} precipitation in Shared Drive...')

if DRIVE_CACHE_AVAILABLE:
    # Scan the country folder for any NetCDF matching the chosen product
    _matching = [
        f for f in os.listdir(COUNTRY_DRIVE_DIR)
        if PRECIP_SOURCE.upper() in f.upper() and f.endswith('.nc')
    ]
    if _matching:
        _precip_filename = _matching[0]
        _precip_local    = os.path.join(LOCAL_CACHE_DIR, _precip_filename)
        if not os.path.exists(_precip_local):
            shutil.copy(os.path.join(COUNTRY_DRIVE_DIR, _precip_filename), _precip_local)
        precip_ds = xr.open_dataset(_precip_local)
        precip_ds = subset_to_bbox(precip_ds, xmin, xmax, ymin, ymax)
        print(f'   ✅ Found: {_precip_filename}')
    else:
        print(f'   ⚠️  No {PRECIP_SOURCE} NetCDF found in Shared Drive/{COUNTRY}/')
        print(f'   Available files: {os.listdir(COUNTRY_DRIVE_DIR)}')
        print(f'   Run FA2 for {COUNTRY} first, or change PRECIP_SOURCE in Cell 0.')

print('\n📊 Input summary:')
print(f'   ERA5 temperature : {"✅ loaded" if era5_lt is not None else "❌ missing"}')
print(f'   Precipitation    : {"✅ " + str(_precip_filename) if precip_ds is not None else "❌ missing"}')

In [ ]:
# @title ### 4) Prepare monthly climate arrays for PyAEZ {"display-mode":"form"}
# @markdown Converts xarray Datasets into the numpy arrays PyAEZ expects:
# @markdown shape (12, nrows, ncols) for monthly data.
# @markdown Also recomputes monthly PET using the Hargreaves equation.

# ── HELPER: resample xarray to monthly mean/sum and extract numpy ─────────────
def to_monthly_numpy(ds, method='mean'):
    """
    Returns a (12, nrows, ncols) numpy array of multi-year monthly means.
    'ds' must have a 'time' dimension.
    For scalar/1-D datasets (country-mean series), returns shape (12,).
    """
    var0 = list(ds.data_vars)[0]
    da   = ds[var0]

    if method == 'mean':
        monthly = da.resample(time='1ME').mean()
    else:
        monthly = da.resample(time='1ME').sum()

    # Group by calendar month and average across years
    climatology = monthly.groupby('time.month').mean('time')
    return climatology.values   # shape: (12,) or (12, lat, lon)

# ── TEMPERATURE ARRAYS ────────────────────────────────────────────────────────
print('Preparing temperature arrays...')

if era5_lt is not None:
    # Long-term ERA5 has only one temperature variable
    _var0 = list(era5_lt.data_vars)[0]
    _da   = era5_lt[_var0]

    # If it's a spatial dataset (lat, lon dims) compute spatial mean for LGP
    _spatial_dims = [d for d in _da.dims if d != 'time']
    if _spatial_dims:
        _da_mean = _da.mean(dim=_spatial_dims)
    else:
        _da_mean = _da

    # Convert K→C if needed
    if float(_da_mean.mean()) > 100:
        _da_mean = _da_mean - 273.15

    # Monthly climatology → shape (12,)
    monthly_resampled  = _da_mean.resample(time='1ME').mean()
    tavg_monthly_1d    = monthly_resampled.groupby('time.month').mean('time').values

    # Estimate tmin/tmax from tavg using a typical diurnal range for the region
    # (approximate — FA3's actual tmin/tmax arrays are more accurate if available)
    DIURNAL_RANGE = 10.0   # °C — typical for East Africa
    tmin_monthly_1d = tavg_monthly_1d - (DIURNAL_RANGE / 2)
    tmax_monthly_1d = tavg_monthly_1d + (DIURNAL_RANGE / 2)

    print(f'   ✅ Monthly Tavg (°C): {np.round(tavg_monthly_1d, 1)}')
    print(f'   ✅ Monthly Tmin (°C): {np.round(tmin_monthly_1d, 1)}')
    print(f'   ✅ Monthly Tmax (°C): {np.round(tmax_monthly_1d, 1)}')
else:
    print('⚠️  ERA5 temperature not available — using East Africa typical values.')
    # Fallback climatology for East Africa (broad average)
    tavg_monthly_1d = np.array([22,23,24,24,22,19,18,19,21,22,22,21], dtype=float)
    tmin_monthly_1d = tavg_monthly_1d - 5.0
    tmax_monthly_1d = tavg_monthly_1d + 5.0

# ── PRECIPITATION ARRAYS ──────────────────────────────────────────────────────
print('\nPreparing precipitation arrays...')

if precip_ds is not None:
    precip_monthly_1d = to_monthly_numpy(precip_ds, method='sum')   # shape (12,) or (12,lat,lon)
    # If spatial, take country mean for scalar PyAEZ inputs
    if precip_monthly_1d.ndim > 1:
        precip_monthly_1d = np.nanmean(precip_monthly_1d, axis=tuple(range(1, precip_monthly_1d.ndim)))
    print(f'   ✅ Monthly precip (mm): {np.round(precip_monthly_1d, 1)}')
else:
    print('⚠️  Precipitation not available — using East Africa typical values.')
    # Typical bimodal East Africa pattern
    precip_monthly_1d = np.array([40,50,90,120,100,30,15,20,50,90,80,45], dtype=float)

# ── HARGREAVES PET ────────────────────────────────────────────────────────────
print('\nComputing monthly PET (Hargreaves)...')

# Ra = extraterrestrial radiation (MJ m-2 day-1) — latitudinal monthly values
# Using country centroid latitude
lat_c = (ymin + ymax) / 2

# Simplified Ra table for tropics (MJ m-2 day-1)
# More accurate would use pyaez.UtilityCalculations.calcRa()
if PYAEZ_AVAILABLE:
    try:
        UC  = UtilityCalculations.UtilityCalculations()
        Ra_monthly = np.array([
            float(UC.calcRa(lat_c, m)) for m in range(1, 13)
        ])
        print(f'   ✅ Ra from PyAEZ UtilityCalculations: {np.round(Ra_monthly, 1)}')
    except Exception as e:
        print(f'   ⚠️  PyAEZ Ra calculation failed ({e}) — using approximation.')
        Ra_monthly = np.full(12, 15.0)
else:
    Ra_monthly = np.full(12, 15.0)   # Approximate for East Africa

# Hargreaves PET (mm/day)
pet_monthly_1d = (
    0.0023
    * Ra_monthly
    * (tavg_monthly_1d + 17.8)
    * np.sqrt(np.maximum(tmax_monthly_1d - tmin_monthly_1d, 0.1))
)
# Convert mm/day → mm/month (approximate: *30)
days_per_month  = np.array([31,28,31,30,31,30,31,31,30,31,30,31])
pet_monthly_mm  = pet_monthly_1d * days_per_month

print(f'   ✅ Monthly PET (mm): {np.round(pet_monthly_mm, 1)}')

# ── WATER BALANCE ─────────────────────────────────────────────────────────────
# P - PET per month: positive = surplus, negative = deficit
water_balance_monthly = precip_monthly_1d - pet_monthly_mm
print(f'\n   Water balance (P-PET, mm): {np.round(water_balance_monthly, 1)}')

In [ ]:
# @title ### 5) Module I — Climate Regime Analysis {"display-mode":"form"}
# @markdown Runs PyAEZ Module I: ClimateRegime.
# @markdown Computes: Thermal Climate Classification, Length of Growing Period (LGP),
# @markdown and moisture zone classification for the selected country.

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# ─────────────────────────────────────────────────────────────────────────────
# PYAEZ ClimateRegime (if available)
# ─────────────────────────────────────────────────────────────────────────────
lgp_value       = None
thermal_class   = None
moisture_class  = None

if PYAEZ_AVAILABLE:
    try:
        # PyAEZ ClimateRegime expects 2-D arrays (nrows, ncols) × 12 months
        # For a single-point (country-mean) analysis we use (1,1) arrays
        _nrows, _ncols = 1, 1
        _mask = np.ones((_nrows, _ncols), dtype=int)

        # Reshape 1-D monthly arrays to (12, 1, 1)
        _tavg = tavg_monthly_1d.reshape(12, _nrows, _ncols)
        _tmin = tmin_monthly_1d.reshape(12, _nrows, _ncols)
        _tmax = tmax_monthly_1d.reshape(12, _nrows, _ncols)
        _prec = precip_monthly_1d.reshape(12, _nrows, _ncols)
        _pet  = pet_monthly_mm.reshape(12, _nrows, _ncols)

        cr = ClimateRegime.ClimateRegime()
        cr.setStudyAreaMask(_mask, no_data_value=0)
        # Note: PyAEZ source has a typo — method is 'setLocationTeperature'
        cr.setLocationTeperature(
            minT_monthly=_tmin,
            maxT_monthly=_tmax,
            meanT_monthly=_tavg
        )
        cr.setMonthlyPrecipitation(_prec)
        cr.setMonthlyPET(_pet)

        # Compute thermal climate
        cr.getThermalClimate()
        thermal_class = int(cr.thermal_climate[0, 0])

        # Compute LGP
        cr.getLGP()
        lgp_value = int(cr.lgp[0, 0])

        # Moisture zone
        cr.getMoistureZone()
        moisture_class = int(cr.moisture_zone[0, 0])

        print('✅ PyAEZ ClimateRegime complete.')

    except Exception as e:
        print(f'⚠️  PyAEZ ClimateRegime error: {e}')
        print('   Falling back to manual calculations.')
        PYAEZ_AVAILABLE = False

# ─────────────────────────────────────────────────────────────────────────────
# MANUAL FALLBACK (always runs for display; also sole path if PyAEZ failed)
# Uses FAO AEZ rules directly — equivalent to what ClimateRegime computes
# ─────────────────────────────────────────────────────────────────────────────

# ── Thermal climate classification (FAO AEZ Table 3.1) ───────────────────────
THERMAL_CLASSES = {
    1: 'Tropics, lowland    (Tavg ≥ 20°C all months)',
    2: 'Tropics, subtropics (Tavg 17–20°C, no frost)',
    3: 'Subtropics, warm    (Tavg 10–20°C, < 3 cold months)',
    4: 'Subtropics, cool    (Tavg 5–15°C, 3–6 cold months)',
    5: 'Temperate, moderate (Tavg 0–10°C, frost present)',
    6: 'Temperate, cool/cold (Tavg < 5°C in winter)',
    7: 'Boreal / Arctic',
}

def classify_thermal(tavg):
    t_min_annual = tavg.min()
    t_mean_annual = tavg.mean()
    cold_months = (tavg < 5).sum()
    if t_min_annual >= 20:                        return 1
    if t_min_annual >= 17 and cold_months == 0:   return 2
    if t_mean_annual >= 15 and cold_months < 3:   return 3
    if t_mean_annual >= 10 and cold_months < 6:   return 4
    if t_mean_annual >= 5:                        return 5
    if t_mean_annual >= 0:                        return 6
    return 7

thermal_class_manual = classify_thermal(tavg_monthly_1d)
if thermal_class is None:
    thermal_class = thermal_class_manual

# ── Length of Growing Period (FAO AEZ approach) ───────────────────────────────
# LGP = number of months where P > 0.5 × PET, converted to days
# (simplified; full PyAEZ version uses daily interpolation)
def calc_lgp(precip_mm, pet_mm):
    growing_months = (precip_mm >= 0.5 * pet_mm).sum()
    # Approximate growing days per month
    return int(growing_months * 30)

lgp_manual = calc_lgp(precip_monthly_1d, pet_monthly_mm)
if lgp_value is None:
    lgp_value = lgp_manual

# ── Moisture zone classification ──────────────────────────────────────────────
MOISTURE_CLASSES = {
    1: 'Hyper-arid    (LGP < 30 days)',
    2: 'Arid          (LGP 30–60 days)',
    3: 'Semi-arid     (LGP 60–120 days)',
    4: 'Sub-humid     (LGP 120–180 days)',
    5: 'Humid         (LGP 180–270 days)',
    6: 'Per-humid     (LGP > 270 days)',
}
def classify_moisture(lgp):
    if lgp < 30:  return 1
    if lgp < 60:  return 2
    if lgp < 120: return 3
    if lgp < 180: return 4
    if lgp < 270: return 5
    return 6

moisture_class_manual = classify_moisture(lgp_value)
if moisture_class is None:
    moisture_class = moisture_class_manual

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print('\n' + '═'*60)
print(f'  CLIMATE REGIME RESULTS — {COUNTRY}')
print('═'*60)
print(f'  Thermal class   : {thermal_class} — {THERMAL_CLASSES.get(thermal_class,"Unknown")}')
print(f'  LGP             : {lgp_value} days/year')
print(f'  Moisture zone   : {moisture_class} — {MOISTURE_CLASSES.get(moisture_class,"Unknown")}')
print(f'  Annual Tavg     : {tavg_monthly_1d.mean():.1f}°C')
print(f'  Annual precip   : {precip_monthly_1d.sum():.0f} mm')
print(f'  Annual PET      : {pet_monthly_mm.sum():.0f} mm')
print(f'  Aridity index   : {precip_monthly_1d.sum()/pet_monthly_mm.sum():.2f} (P/PET)')
print('═'*60)

In [ ]:
# @title ### 6) Crop suitability assessment {"display-mode":"form"}
# @markdown Evaluates which crops are thermally and moisture-suitable for the country
# @markdown using FAO AEZ thermal and water requirements.
# @markdown Crops relevant to East Africa: Maize, Sorghum, Wheat, Millet, Rice, Cassava.

# ── CROP REQUIREMENTS (FAO AEZ) ───────────────────────────────────────────────
# Source: FAO GAEZ v4 crop parameter tables
CROPS = {
    'Maize': {
        'tavg_min': 10.0, 'tavg_max': 35.0,
        'tmin_abs': 5.0,  'tmax_abs': 40.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 400, 'precip_opt': 700,
        'description': 'Most important food crop in East Africa.'
    },
    'Sorghum': {
        'tavg_min': 12.0, 'tavg_max': 37.0,
        'tmin_abs': 6.0,  'tmax_abs': 42.0,
        'lgp_min': 75,    'lgp_opt': 120,
        'precip_min': 250, 'precip_opt': 500,
        'description': 'Drought-tolerant. Suited to semi-arid zones.'
    },
    'Wheat': {
        'tavg_min': 5.0,  'tavg_max': 25.0,
        'tmin_abs': 0.0,  'tmax_abs': 32.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 300, 'precip_opt': 600,
        'description': 'Higher elevations (Ethiopia highlands).'
    },
    'Millet (Pearl)': {
        'tavg_min': 14.0, 'tavg_max': 38.0,
        'tmin_abs': 8.0,  'tmax_abs': 44.0,
        'lgp_min': 60,    'lgp_opt': 100,
        'precip_min': 200, 'precip_opt': 400,
        'description': 'Most drought-tolerant cereal. Suited to arid margins.'
    },
    'Rice (Upland)': {
        'tavg_min': 18.0, 'tavg_max': 35.0,
        'tmin_abs': 12.0, 'tmax_abs': 38.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 800, 'precip_opt': 1200,
        'description': 'Humid lowlands and irrigated areas.'
    },
    'Cassava': {
        'tavg_min': 18.0, 'tavg_max': 35.0,
        'tmin_abs': 10.0, 'tmax_abs': 38.0,
        'lgp_min': 120,   'lgp_opt': 240,
        'precip_min': 500, 'precip_opt': 1000,
        'description': 'Resilient root crop. Tolerates poor soils.'
    },
}

# ── ASSESS SUITABILITY ────────────────────────────────────────────────────────
tavg_annual   = tavg_monthly_1d.mean()
tmin_annual   = tmin_monthly_1d.min()
tmax_annual   = tmax_monthly_1d.max()
precip_annual = precip_monthly_1d.sum()

results = []

for crop, req in CROPS.items():
    # Thermal suitability
    thermal_ok = (
        req['tavg_min'] <= tavg_annual <= req['tavg_max']
        and tmin_annual >= req['tmin_abs']
        and tmax_annual <= req['tmax_abs'] + 3  # small tolerance
    )

    # LGP suitability
    lgp_ok  = lgp_value >= req['lgp_min']
    lgp_opt = lgp_value >= req['lgp_opt']

    # Water suitability
    water_ok  = precip_annual >= req['precip_min']
    water_opt = precip_annual >= req['precip_opt']

    # Overall suitability
    if thermal_ok and lgp_ok and water_ok:
        if lgp_opt and water_opt:
            suitability = 'High'
            score = 3
        else:
            suitability = 'Moderate'
            score = 2
    elif thermal_ok and (lgp_ok or water_ok):
        suitability = 'Marginal'
        score = 1
    else:
        suitability = 'Not suitable'
        score = 0

    results.append({
        'Crop':         crop,
        'Suitability':  suitability,
        'Score':        score,
        'Thermal OK':   '✅' if thermal_ok else '❌',
        'LGP OK':       '✅' if lgp_ok else '❌',
        'Water OK':     '✅' if water_ok else '❌',
        'Description':  req['description'],
    })

df_suitability = pd.DataFrame(results).sort_values('Score', ascending=False)

print(f'\nCROP SUITABILITY ASSESSMENT — {COUNTRY}')
print(f'  Tavg={tavg_annual:.1f}°C  Tmin={tmin_annual:.1f}°C  Tmax={tmax_annual:.1f}°C')
print(f'  LGP={lgp_value} days  Precip={precip_annual:.0f}mm')
print()
from IPython.display import display
display(df_suitability[['Crop','Suitability','Thermal OK','LGP OK','Water OK','Description']])

In [ ]:
# @title ### 7) Results visualisation {"display-mode":"form"}
# @markdown Plots the monthly climate profile, water balance, and crop suitability summary.

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Focus Area 4 — Agro-Ecological Zoning\n{COUNTRY}  |  {LT_START}–{LT_END}',
    fontsize=14, fontweight='bold', y=0.98
)

# ── Panel 1: Temperature profile ──────────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
ax1.fill_between(range(12), tmin_monthly_1d, tmax_monthly_1d, alpha=0.2, color='tomato', label='Tmin–Tmax range')
ax1.plot(range(12), tavg_monthly_1d, 'o-', color='tomato', lw=2, ms=5, label='Tavg')
ax1.axhline(10, color='steelblue', ls='--', lw=0.8, label='10°C (crop base)')
ax1.axhline(0,  color='black',     ls='--', lw=0.8, label='0°C (frost)')
ax1.set_xticks(range(12)); ax1.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('°C'); ax1.set_title('Monthly Temperature')
ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)

# ── Panel 2: Precipitation vs PET ─────────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
ax2.bar(range(12), precip_monthly_1d, color='steelblue', alpha=0.8, label='Precipitation')
ax2.plot(range(12), pet_monthly_mm, 'o-', color='tomato', lw=2, ms=5, label='PET (Hargreaves)')
ax2.set_xticks(range(12)); ax2.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('mm'); ax2.set_title('Precipitation vs PET')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3, axis='y')

# ── Panel 3: Water balance ────────────────────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
colors_wb = ['steelblue' if v >= 0 else 'tomato' for v in water_balance_monthly]
ax3.bar(range(12), water_balance_monthly, color=colors_wb, alpha=0.85)
ax3.axhline(0, color='black', lw=1)
ax3.set_xticks(range(12)); ax3.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('mm'); ax3.set_title('Water Balance (P − PET)')
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: Crop suitability bar chart ───────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
_suit_colors = {'High':'#2e7d52','Moderate':'#c87c1a','Marginal':'#d4a017','Not suitable':'#b5341a'}
ax4.barh(
    df_suitability['Crop'],
    df_suitability['Score'],
    color=[_suit_colors[s] for s in df_suitability['Suitability']],
    alpha=0.85
)
ax4.set_xlim(0, 3.5)
ax4.set_xticks([0,1,2,3])
ax4.set_xticklabels(['Not\nSuitable','Marginal','Moderate','High'], fontsize=8)
ax4.set_title('Crop Suitability Score')
ax4.grid(True, alpha=0.3, axis='x')

# ── Panel 5: AEZ summary text card ────────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
ax5.axis('off')
summary_text = (
    f"AEZ SUMMARY\n"
    f"{'─'*28}\n"
    f"Country       : {COUNTRY}\n"
    f"Thermal class : {thermal_class}\n"
    f"  {THERMAL_CLASSES.get(thermal_class,'?')}\n\n"
    f"Moisture zone : {moisture_class}\n"
    f"  {MOISTURE_CLASSES.get(moisture_class,'?')}\n\n"
    f"LGP           : {lgp_value} days/yr\n"
    f"Annual Tavg   : {tavg_annual:.1f}°C\n"
    f"Annual Precip : {precip_annual:.0f} mm\n"
    f"Annual PET    : {pet_monthly_mm.sum():.0f} mm\n"
    f"Aridity (P/PET): {precip_annual/pet_monthly_mm.sum():.2f}\n\n"
    f"Precip source : {PRECIP_SOURCE}\n"
    f"Period        : {LT_START}–{LT_END}"
)
ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes,
         va='top', fontsize=9, fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f4f6fb', alpha=0.8))

# ── Panel 6: Growing period moisture indicator ─────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
growing = (precip_monthly_1d >= 0.5 * pet_monthly_mm).astype(int)
ax6.bar(range(12), growing, color=['#2e7d52' if g else '#e8ddd0' for g in growing], alpha=0.85)
ax6.set_xticks(range(12)); ax6.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax6.set_yticks([0, 1]); ax6.set_yticklabels(['Dry', 'Growing'])
ax6.set_title(f'Monthly Growing Conditions\n(LGP ≈ {lgp_value} days/year)')
ax6.grid(True, alpha=0.3, axis='y')

# ── Panel 7–9: Long-term temperature trend ────────────────────────────────────
if era5_lt is not None:
    ax7 = fig.add_subplot(3, 1, 3)
    _var0 = list(era5_lt.data_vars)[0]
    _da   = era5_lt[_var0]
    _sp   = [d for d in _da.dims if d != 'time']
    _vals = _da.mean(dim=_sp).values if _sp else _da.values
    if float(np.nanmean(_vals)) > 100:
        _vals = _vals - 273.15
    _times = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in era5_lt.time.values])
    ax7.plot(_times, _vals, color='tomato', lw=1.2, alpha=0.8)
    # 12-month rolling mean
    _rolling = pd.Series(_vals, index=_times).rolling(12, center=True).mean()
    ax7.plot(_rolling.index, _rolling.values, color='darkred', lw=2, label='12-month mean')
    ax7.set_ylabel('Temperature (°C)')
    ax7.set_title(f'Long-term ERA5 Temperature — {COUNTRY}  ({LT_START}–{LT_END})')
    ax7.legend(fontsize=8); ax7.grid(True, alpha=0.3)

plt.tight_layout()

# Save
_plot_path = os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')
plt.savefig(_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to: {_plot_path}')

# Also save the suitability table
_csv_path = os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')
df_suitability.to_csv(_csv_path, index=False)
print(f'✅ Suitability table saved to: {_csv_path}')

In [ ]:
# @title ### 8) Cache results to Shared Drive {"display-mode":"form"}
# @markdown Uploads the AEZ outputs to the Shared Drive so other participants
# @markdown running the same country can access pre-computed results.

from google.colab import files

print(f'☁️  Caching FA4 results to Shared Drive/{COUNTRY}...')

_outputs = [
    (f'FA4_AEZ_{COUNTRY}.png',              os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')),
    (f'FA4_crop_suitability_{COUNTRY}.csv', os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')),
]

for drive_name, local_path in _outputs:
    if os.path.exists(local_path):
        cache_to_drive(local_path, drive_name)
    else:
        print(f'   ⚠️  Skipping {drive_name} — file not found (run Cell 7 first)')

print('\n📥 Downloading results to your local machine...')
for _, local_path in _outputs:
    if os.path.exists(local_path):
        files.download(local_path)

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print('\n' + '═'*60)
print(f'  FOCUS AREA 4 COMPLETE — {COUNTRY}')
print('═'*60)
print(f'  Thermal class  : {thermal_class} — {THERMAL_CLASSES.get(thermal_class,"?")[:35]}')
print(f'  Moisture zone  : {moisture_class} — {MOISTURE_CLASSES.get(moisture_class,"?")[:35]}')
print(f'  LGP            : {lgp_value} days/year')
print()

_high  = df_suitability[df_suitability['Suitability']=='High']['Crop'].tolist()
_mod   = df_suitability[df_suitability['Suitability']=='Moderate']['Crop'].tolist()
_marg  = df_suitability[df_suitability['Suitability']=='Marginal']['Crop'].tolist()
_none  = df_suitability[df_suitability['Suitability']=='Not suitable']['Crop'].tolist()

if _high:  print(f'  ✅ High suitability     : {", ".join(_high)}')
if _mod:   print(f'  🟡 Moderate suitability : {", ".join(_mod)}')
if _marg:  print(f'  🟠 Marginal suitability : {", ".join(_marg)}')
if _none:  print(f'  ❌ Not suitable         : {", ".join(_none)}')
print('═'*60)